# 필요한 의미만 해석하는 맥락 선택

연구용 프로토타입. 운영 변경 없음. 한국어 패턴 함수와 가상 계획 기록이며 실제 LLM 평가가 아니다. 전체 해설은 research.html. 노트북은 미실행 재현 문서다. phase05 폴더에서 실행한다.


In [ ]:
import json
from pathlib import Path
base = Path.cwd()
assert (base / 'protocol.json').exists()
r = json.loads((base / 'results.json').read_text(encoding='utf-8'))
r['summary']


## 역방향 의존성

현재 질문 필드에서 시작한다. 상수 대입은 해당 필드의 이전 값 의존성을 끝내고, 상대 변경은 유지하며, 복사는 그 시점의 다른 필드로 요구를 옮긴다. 미해석 문장은 전체 영향으로 간주해 상세 해석하며, 해결하지 못하면 전체 원문으로 복원한다.

전체 상세 해석과 동일한 결과를 672개 질문에서 얻었다. 상세 단계 호출은 기본 3,140→224, 낯선 표현 조합 2,562→240이었다. 로컬 함수 호출 수이며 LLM 호출 측정이 아니다.


## 조건부 정확성

각 문장 전이에 대해 $F_i(\gamma(a))\subseteq\gamma(F_i^\#(a))$가 성립하면, 추상 결과에는 실제 결과가 포함된다. 질문 필드의 결과가 한 값이면 정의된 의미론 안에서는 그 값으로 확정할 수 있다.

이 전제는 자연어 해석의 정확성을 보장해 주지 않는다. 실제 실행 완료가 아니라 계획 값을 다룬다. 스냅샷 해시도 기록의 byte 동일성만 확인하며 최신 head나 의미를 보장하지 않는다.


In [ ]:
r['fault_checks'], [x for x in r['scaling'] if x['length']==4096]


## 상대 변경 구간의 최소 비용

중간 읽기·다른 쓰기가 없는 같은 필드의 정수 이동에서, 원문 부분집합의 합이 원래 합과 같으면 최종 값이 보존된다.

$$D_i(s)=\min(D_{i-1}(s),D_{i-1}(s-d_i)+c_i)$$

목표는 $D_m(\sum_i d_i)$. 구간 내부의 정확한 최적해이며 전체 문맥 최소 비용은 아니다. 중간 값을 다른 필드가 읽으면 그 지점에서 반드시 구간을 끊는다.


In [ ]:
v2 = json.loads((base / 'revision02/reduction_results.json').read_text(encoding='utf-8'))
{k:v for k,v in v2.items() if k!='examples'}


## 부정 결과도 보존

낯선 표현 조합은 288개 질문 중 36개에서만 단일 답 판정, 240개 전체 원문 복원, 12개 미지원 질문이었다. 상세 해석 원문 bytes까지 더한 절감은 필요한 곳만 해석해도 4.31%였다.

첫 상쇄 실험은 '앞으로'를 인식하지 못해 전체 원문 복원으로 빠졌으며 실제 절감 0이었다. 수정 전 결과와 원인을 보존했다. 수정 버전에서 단일 답 판정 상태의 실제 상쇄와 11개 연산 의미를 추가 확인했다. 43,923개 질문 비교에서 답을 보존했고 2,936개가 짧아졌지만, 추가 절감은 그 연산 조합에서 2.42%였다. 기존 672개 대화 질문에는 이득이 없고 중복 상세 해석 비용이 있었다.


In [ ]:
# 공개된 재현 명령; 네트워크·운영 데이터·LLM 호출 없음
import subprocess, sys
subprocess.run([sys.executable, str(base/'study.py'), '--run'], cwd=base, check=True)
subprocess.run([sys.executable, str(base/'audit.py')], cwd=base, check=True)
subprocess.run([sys.executable, str(base/'revision02/reduction_study.py'), '--run'], cwd=base, check=True)


## 관련 문헌

- [Cousot & Cousot 1977](https://cs.nyu.edu/~pcousot/COUSOTpapers/POPL77.shtml): 보수적인 추상 의미.
- [Clarke et al. 2000](https://www.cs.cmu.edu/~emc/papers/Conference%20Papers/Counterexample-guided%20Abstraction%20Refinement.pdf): 추상화의 단계적 정밀화.

문헌의 이론을 참고한 제한된 연구 모형이다. 실제 자연어 이해, 실제 모델 답변, 전체 토큰·지연, 보호 메모리 정책은 검증하지 않았다.
